# Latent Factor Models — Exploratory Analysis

This notebook explores the data and model outputs interactively.
Run `main.py` first to download data and train models.

Topics covered:
1. Return distribution and coverage statistics
2. Characteristics cross-sectional distributions
3. PCA variance explained
4. Factor time series (AE vs CAE)
5. Nonlinear contribution decomposition (CAE)
6. Factor portfolio performance

In [1]:
import sys
sys.path.insert(0, '..')

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## 1. Load cached data

In [2]:
CACHE = '../data/raw/data_cache.pkl'

with open(CACHE, 'rb') as f:
    data = pickle.load(f)

splits  = data['splits']
returns = data['returns']     # (T, N) excess returns
chars   = data['chars']       # (N, T, P)
dates   = data['dates']
tickers = data['tickers']

print(f'Full sample: {returns.shape[0]} months × {returns.shape[1]} stocks')
print(f'Date range:  {dates[0].date()} → {dates[-1].date()}')

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/data_cache.pkl'

## 2. Return distribution

In [ ]:
ret_vals = returns.values.flatten()
ret_vals = ret_vals[~np.isnan(ret_vals)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(ret_vals, bins=200, range=(-0.5, 0.5), color='steelblue', alpha=0.7)
axes[0].set_title('Monthly Excess Return Distribution')
axes[0].set_xlabel('Monthly Return')
axes[0].set_ylabel('Frequency')

# Coverage: % stocks available per month
coverage = returns.notna().mean(axis=1) * 100
axes[1].plot(returns.index, coverage, lw=0.8)
axes[1].set_title('Stock Coverage Over Time (% of universe)')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('% Stocks Available')

plt.tight_layout()
plt.show()

print(f'Mean excess return: {ret_vals.mean():.4f}')
print(f'Std  excess return: {ret_vals.std():.4f}')
print(f'Skewness:           {pd.Series(ret_vals).skew():.3f}')
print(f'Kurtosis:           {pd.Series(ret_vals).kurtosis():.3f}')

## 3. Characteristics distributions

In [ ]:
char_names = ['mom1', 'mom6', 'mom12', 'vol12', 'beta12']
P = len(char_names)

fig, axes = plt.subplots(1, P, figsize=(15, 3))

for p, name in enumerate(char_names):
    vals = chars[:, :, p].flatten()
    vals = vals[~np.isnan(vals)]
    axes[p].hist(vals, bins=100, color='darkorange', alpha=0.7)
    axes[p].set_title(name)
    axes[p].set_xlabel('Rank-normalised value')

plt.suptitle('Cross-sectional Characteristic Distributions (rank-normalised to [-1,1])')
plt.tight_layout()
plt.show()

## 4. PCA variance explained

In [ ]:
from sklearn.decomposition import PCA

train_ret = splits['train']['returns'].values
col_means = np.nanmean(train_ret, axis=0)
filled    = np.where(np.isnan(train_ret), col_means[None,:], train_ret)

pca_full = PCA(n_components=20).fit(filled)
var_exp  = pca_full.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, len(var_exp)+1), var_exp * 100, color='steelblue')
ax.plot(range(1, len(var_exp)+1), np.cumsum(var_exp)*100,
        'ro-', markersize=4, label='Cumulative')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Variance Explained (%)')
ax.set_title('PCA Scree Plot (Training Data)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Load saved summary table

In [ ]:
summary = pd.read_csv('../results/summary_table.csv')
summary

In [ ]:
# Heatmap: Total R2 by Model and K
df_num = summary.copy()
df_num['Total_R2'] = pd.to_numeric(df_num['Total_R2'], errors='coerce')
df_num['Sharpe']   = pd.to_numeric(df_num['Sharpe'],   errors='coerce')

pivot = df_num.pivot_table(index='Model', columns='K',
                            values='Total_R2', aggfunc='mean')

fig, ax = plt.subplots(figsize=(7, 3))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='RdYlGn', ax=ax)
ax.set_title('Total R² — Test Set (2020–2024)')
plt.tight_layout()
plt.show()